# LPG Flow Prediction
## IOCL Internship – Naphtha Stabilizer Column

### Objective

Develop a machine learning model to predict **LPG Flow (Quantity)** using process variables from the Naphtha Stabilizer Column.

This notebook focuses only on predicting LPG Flow.

The Weathering Temperature prediction notebook is treated as a separate module.

Later, both models will be combined to optimize:

- LPG Quantity (Flow)
- LPG Quality (Weathering Temperature)

---

### Machine Learning Workflow

For every experiment, follow:

Hypothesis
↓
Reason
↓
Experiment
↓
Train
↓
Compare
↓
Conclusion

Only one hypothesis will be tested at a time.

In [4]:
# Data Handling
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Settings
pd.set_option("display.max_columns", None)
plt.style.use("default")
sns.set_theme(style="whitegrid")

In [5]:
df = pd.read_csv("I:\Projects\iocl\data\df_clean.csv")

print(df.shape)
df.head()
df.info()

(458, 24)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 458 entries, 0 to 457
Data columns (total 24 columns):
 #   Column                                                  Non-Null Count  Dtype  
---  ------                                                  --------------  -----  
 0   DATE                                                    458 non-null    object 
 1   Stabilizer Feed T                                       458 non-null    float64
 2   Stabilizer Feed Flow                                    458 non-null    float64
 3   Stabilizer Top P                                        458 non-null    float64
 4   Stabilizer Reflux Drum T                                458 non-null    float64
 5   Stabilized Naphtha Flow                                 458 non-null    float64
 6   Stabilizer Reflux Flow                                  458 non-null    float64
 7   HGO CR Flow                                             458 non-null    float64
 8   HGO CR to reboiler Flow       

<>:1: SyntaxWarning: invalid escape sequence '\P'
<>:1: SyntaxWarning: invalid escape sequence '\P'
C:\Users\Vansh Pandey\AppData\Local\Temp\ipykernel_8944\2555388580.py:1: SyntaxWarning: invalid escape sequence '\P'
  df = pd.read_csv("I:\Projects\iocl\data\df_clean.csv")


# Phase 1 — Data Cleaning

## Objective

Prepare a clean dataset for machine learning by removing redundant and unreliable features.

### Cleaning Steps

- Remove duplicate sensor columns.
- Remove the faulty sensor tag.
- Remove the `DATE` column.
- Keep the target (`LPG Flow`) in the dataset for now.
- Keep `Wthr` for now (it will simply not be used as an input feature).

This preprocessing is performed once and will be used for all subsequent experiments.

In [6]:
# Create a copy of the original dataset
df_clean = df.copy()
from scipy import stats

# ---------------------------------------------------
# Remove duplicate sensor columns
# ---------------------------------------------------
duplicate_columns = [
    "Stabilizer Top T.1",
    "Stab. 3rd Tray.1",
    "Bot. Naphtha Reboiler Outlet Temp(TI-1908 & TI-1909).1"
]

# ---------------------------------------------------
# Remove unnecessary columns
# ---------------------------------------------------
remove_columns = [
    "DATE",
    "Off Spec LPG flow (tag Faulty)"
]

# Drop columns
df_clean = df_clean.drop(columns=duplicate_columns + remove_columns)

# ---------------------------------------------------
# Verify cleaned dataset
# ---------------------------------------------------
print("=" * 50)
print("Clean Dataset")
print("=" * 50)

print(f"Rows    : {df_clean.shape[0]}")
print(f"Columns : {df_clean.shape[1]}")

print("\nRemaining Columns:\n")
print(df_clean.columns.tolist())

Clean Dataset
Rows    : 458
Columns : 19

Remaining Columns:

['Stabilizer Feed T', 'Stabilizer Feed Flow', 'Stabilizer Top P', 'Stabilizer Reflux Drum T', 'Stabilized Naphtha Flow', 'Stabilizer Reflux Flow', 'HGO CR Flow', 'HGO CR to reboiler Flow', 'LPG Flow', 'Stabilizer Top T', 'Stabilizer Bottom T', 'Stabiliser bottom level', 'Stabilier bottom pressure', 'Stab. 3rd Tray', 'HGO CR Reboiler Inlet Temp( TI-1914)', 'Bottom Reboiler Naphtha Inlet Temp( TI-1907)', 'Bot. Naphtha Reboiler Outlet Temp(TI-1908 & TI-1909)', 'Off Spec LPG from CRU inlet pressure', 'Wthr']


# Phase 2 — Baseline Random Forest Model

## Objective

Establish a baseline machine learning model for predicting **LPG Flow** using the cleaned refinery process data.

## Hypothesis

The cleaned operating variables of the Naphtha Stabilizer Column contain sufficient information to predict LPG Flow with reasonable accuracy.

## Methodology

- Use the cleaned dataset (`df_clean`).
- Input Features: All process variables except `LPG Flow` and `Wthr`.
- Target Variable: `LPG Flow`.
- Train-Test Split: 80% / 20%.
- Model: Random Forest Regressor (default parameters).
- Evaluation Metrics:
  - Mean Absolute Error (MAE)
  - Root Mean Squared Error (RMSE)
  - R² Score

This baseline will be used as the reference for all future feature engineering and model improvements.

In [8]:
# ==========================================================
# Baseline Random Forest Model
# ==========================================================

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import pandas as pd

# -------------------------
# Features and Target
# -------------------------
X = df_clean.drop(columns=["LPG Flow", "Wthr"])
y = df_clean["LPG Flow"]

# -------------------------
# Train-Test Split
# -------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

# -------------------------
# Train Model
# -------------------------
baseline_rf = RandomForestRegressor(random_state=42)

baseline_rf.fit(X_train, y_train)

# -------------------------
# Prediction
# -------------------------
y_pred = baseline_rf.predict(X_test)

# -------------------------
# Evaluation
# -------------------------
mae = mean_absolute_error(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred) ** 0.5
r2 = r2_score(y_test, y_pred)

print("=" * 60)
print("Baseline Random Forest Results")
print("=" * 60)
print(f"MAE  : {mae:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"R²   : {r2:.4f}")

# -------------------------
# Feature Importance
# -------------------------
importance = (
    pd.DataFrame({
        "Feature": X.columns,
        "Importance": baseline_rf.feature_importances_
    })
    .sort_values("Importance", ascending=False)
    .reset_index(drop=True)
)

print("\nTop 10 Important Features")
display(importance.head(10))

Baseline Random Forest Results
MAE  : 2.7091
RMSE : 3.6398
R²   : 0.7320

Top 10 Important Features


,Feature,Importance
0,Stabilizer Top P,0.179363
1,Stabilizer Feed Flow,0.151573
2,Stabilizer Reflux Drum T,0.139605
3,Stabiliser bottom level,0.139274
4,HGO CR Flow,0.059502
5,Stabilizer Feed T,0.051782
6,Stabilizer Top T,0.046589
7,Stabilizer Reflux Flow,0.042058
8,Stabilier bottom pressure,0.036070
9,HGO CR Reboiler Inlet Temp( TI-1914),0.033693


# Experiment 5 — Hyperparameter Tuning

## Hypothesis

The baseline Random Forest uses default parameters.

Optimizing key hyperparameters may improve prediction performance without
changing the dataset or engineered features.

---

## Parameters Tuned

- Number of Trees
- Maximum Tree Depth
- Minimum Samples Split
- Minimum Samples Leaf

---

## Baseline

Random Forest (Default)

R² = 0.7320

---

## Success Criterion

If the tuned Random Forest improves prediction beyond the baseline,
the tuned model becomes the new baseline.

In [9]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

TARGET = "LPG Flow"

X = df_clean.drop(columns=["LPG Flow", "Wthr"])
y = df_clean[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [10, 20, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4]
}

grid = GridSearchCV(
    RandomForestRegressor(random_state=42),
    param_grid,
    cv=5,
    scoring="r2",
    n_jobs=-1
)

print("Searching best parameters...\n")

grid.fit(X_train, y_train)

best_model = grid.best_estimator_

pred = best_model.predict(X_test)

mae = mean_absolute_error(y_test, pred)
rmse = np.sqrt(mean_squared_error(y_test, pred))
r2 = r2_score(y_test, pred)

print("="*70)
print("TUNED RANDOM FOREST")
print("="*70)

print("Best Parameters")
print(grid.best_params_)

print("\nPerformance")
print("-"*30)
print(f"MAE  : {mae:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"R²   : {r2:.4f}")

print("\nBaseline R² : 0.7320")

if r2 > 0.7320:
    print("\n✅ Hypothesis Accepted")
else:
    print("\n❌ Hypothesis Rejected")

Searching best parameters...

TUNED RANDOM FOREST
Best Parameters
{'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 100}

Performance
------------------------------
MAE  : 2.7091
RMSE : 3.6398
R²   : 0.7320

Baseline R² : 0.7320

❌ Hypothesis Rejected


# Experiment 6 — Random Forest vs XGBoost

## Hypothesis

XGBoost may model nonlinear refinery relationships better than Random Forest.

The same feature set will be used.

Only the learning algorithm changes.

---

## Models Compared

Tuned Random Forest

vs

XGBoost

---

## Success Criterion

The model with higher R² becomes the final prediction model.

In [10]:
from xgboost import XGBRegressor

TARGET = "LPG Flow"

X = df_clean.drop(columns=["LPG Flow","Wthr"])
y = df_clean[TARGET]

X_train,X_test,y_train,y_test=train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

model = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42
)

print("="*70)
print("XGBOOST")
print("="*70)

model.fit(X_train,y_train)

pred=model.predict(X_test)

mae=mean_absolute_error(y_test,pred)
rmse=np.sqrt(mean_squared_error(y_test,pred))
r2=r2_score(y_test,pred)

print(f"MAE  : {mae:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"R²   : {r2:.4f}")

print("\nCompare against tuned Random Forest.")

XGBOOST
MAE  : 2.7158
RMSE : 3.7143
R²   : 0.7209

Compare against tuned Random Forest.


# Experiment 7 — 5-Fold Cross Validation

## Hypothesis

The final Random Forest model should produce consistent performance across
multiple train-test splits rather than depending on a single random split.

Cross-validation provides a more reliable estimate of real-world performance.

---

## Model

Random Forest Regressor

---

## Feature Set

All Original Features

---

## Validation

5-Fold Cross Validation

Metrics

- R²
- MAE
- RMSE

---

## Baseline

Single Train-Test Split

R² = 0.7320

---

## Success Criterion

If the average cross-validation performance remains close to the single
train-test performance, the model is considered stable and suitable for
deployment.

In [11]:
# ======================================================
# Experiment 7
# 5-Fold Cross Validation
# ======================================================

from sklearn.model_selection import KFold, cross_validate
from sklearn.ensemble import RandomForestRegressor
import numpy as np

TARGET = "LPG Flow"

X = df_clean.drop(columns=["LPG Flow", "Wthr"])
y = df_clean[TARGET]

# -----------------------------------
# Final Model
# -----------------------------------

model = RandomForestRegressor(
    random_state=42
)

# If tuning improved the model,
# replace the above with:
#
# model = RandomForestRegressor(
#     n_estimators=...,
#     max_depth=...,
#     min_samples_split=...,
#     min_samples_leaf=...,
#     random_state=42
# )

# -----------------------------------
# 5-Fold CV
# -----------------------------------

cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scores = cross_validate(
    model,
    X,
    y,
    cv=cv,
    scoring={
        "r2": "r2",
        "mae": "neg_mean_absolute_error",
        "rmse": "neg_root_mean_squared_error"
    },
    n_jobs=-1
)

# -----------------------------------
# Results
# -----------------------------------

print("=" * 70)
print("5-FOLD CROSS VALIDATION")
print("=" * 70)

print(f"Mean R²   : {scores['test_r2'].mean():.4f}")
print(f"Std  R²   : {scores['test_r2'].std():.4f}")

print()

print(f"Mean MAE  : {-scores['test_mae'].mean():.4f}")
print(f"Mean RMSE : {-scores['test_rmse'].mean():.4f}")

print()

print("Individual Fold R²")
print("-" * 30)

for i, score in enumerate(scores["test_r2"], start=1):
    print(f"Fold {i}: {score:.4f}")

print()

print("=" * 70)

if scores["test_r2"].mean() >= 0.70:
    print("✅ Model is highly stable across folds.")
elif scores["test_r2"].mean() >= 0.60:
    print("✅ Model is reasonably stable.")
elif scores["test_r2"].mean() >= 0.50:
    print("⚠️ Moderate stability.")
else:
    print("❌ Model is unstable across folds.")

5-FOLD CROSS VALIDATION
Mean R²   : 0.6711
Std  R²   : 0.0415

Mean MAE  : 2.9863
Mean RMSE : 4.1253

Individual Fold R²
------------------------------
Fold 1: 0.7188
Fold 2: 0.6249
Fold 3: 0.6889
Fold 4: 0.6184
Fold 5: 0.7042

✅ Model is reasonably stable.


# Experiment 8 — Permutation Feature Importance

## Hypothesis

Some process variables may contribute very little to LPG Flow prediction.

Removing low-importance variables may reduce noise and improve model
generalization.

Unlike impurity-based feature importance, permutation importance measures the
actual decrease in model performance when a feature is randomly shuffled.

---

## Procedure

1. Train the final Random Forest model.

2. Calculate permutation importance.

3. Rank all features.

4. Remove the least important features.

5. Retrain the model.

6. Compare against the current baseline.

---

## Success Criterion

If removing the lowest-importance features improves model performance,
the hypothesis is accepted.

Otherwise, all features will be retained.

In [12]:
from sklearn.inspection import permutation_importance

TARGET = "LPG Flow"

X = df_clean.drop(columns=["LPG Flow","Wthr"])
y = df_clean[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

model = RandomForestRegressor(random_state=42)

model.fit(X_train, y_train)

result = permutation_importance(
    model,
    X_test,
    y_test,
    n_repeats=20,
    random_state=42,
    scoring="r2"
)

importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": result.importances_mean
})

importance = importance.sort_values(
    "Importance",
    ascending=False
).reset_index(drop=True)

print("="*60)
print("PERMUTATION FEATURE IMPORTANCE")
print("="*60)

display(importance)

PERMUTATION FEATURE IMPORTANCE


,Feature,Importance
0,Stabilizer Top P,0.302773
1,Stabilizer Feed Flow,0.234116
2,Stabiliser bottom level,0.206697
3,Stabilizer Reflux Drum T,0.128864
4,HGO CR Flow,0.060958
5,Stabilizer Reflux Flow,0.030273
6,Stabilier bottom pressure,0.030257
7,HGO CR Reboiler Inlet Temp( TI-1914),0.026536
8,Stabilizer Feed T,0.024550
9,Stabilizer Top T,0.023404


# Experiment 9 — Remove Low Importance Features

## Hypothesis

Some process variables contribute almost nothing to LPG Flow prediction.

Removing these variables may reduce model complexity and improve
generalization.

The lowest permutation importance features were removed and the Random
Forest model was retrained.

---

## Removed Features

- Bot. Naphtha Reboiler Outlet Temp(TI-1908 & TI-1909)
- Bottom Reboiler Naphtha Inlet Temp( TI-1907)
- Stab. 3rd Tray
- Stabilizer Bottom T

---

## Baseline

Random Forest

All Features

R² = 0.7320

---

## Success Criterion

If prediction performance improves after removing these variables,
the hypothesis will be accepted.

Otherwise all original features will be retained.

In [7]:
# ======================================================
# Experiment 9
# Remove Lowest Importance Features
# ======================================================

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import numpy as np

TARGET = "LPG Flow"

drop_features = [
    "Bot. Naphtha Reboiler Outlet Temp(TI-1908 & TI-1909)",
    "Bottom Reboiler Naphtha Inlet Temp( TI-1907)",
    "Stab. 3rd Tray",
    "Stabilizer Bottom T"
]

X = df_clean.drop(columns=drop_features + ["LPG Flow", "Wthr"])
print(X.columns.tolist())
y = df_clean[TARGET]

print("=" * 70)
print("MODEL : Random Forest (Low Importance Features Removed)")
print("=" * 70)

print(f"Features Used    : {X.shape[1]}")
print(f"Total Samples    : {len(df_clean)}")

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print(f"Training Samples : {len(X_train)}")
print(f"Testing Samples  : {len(X_test)}")

model = RandomForestRegressor(
    random_state=42
)

print("\nTraining Random Forest...")

model.fit(X_train, y_train)

pred = model.predict(X_test)

mae = mean_absolute_error(y_test, pred)
rmse = np.sqrt(mean_squared_error(y_test, pred))
r2 = r2_score(y_test, pred)

print("\nPerformance")
print("-" * 35)
print(f"MAE  : {mae:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"R²   : {r2:.4f}")

print("\n")
print("=" * 70)
print("BASELINE COMPARISON")
print("=" * 70)

print("Previous Best")
print("MAE  : 2.7091")
print("RMSE : 3.6398")
print("R²   : 0.7320")

print()

print("Current Model")
print(f"MAE  : {mae:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"R²   : {r2:.4f}")

print()

if r2 > 0.7320:
    print("✅ Hypothesis Accepted")
    print("Removing low-importance features improved prediction.")
else:
    print("❌ Hypothesis Rejected")
    print("All original features will be retained.")

['Stabilizer Feed T', 'Stabilizer Feed Flow', 'Stabilizer Top P', 'Stabilizer Reflux Drum T', 'Stabilized Naphtha Flow', 'Stabilizer Reflux Flow', 'HGO CR Flow', 'HGO CR to reboiler Flow', 'Stabilizer Top T', 'Stabiliser bottom level', 'Stabilier bottom pressure', 'HGO CR Reboiler Inlet Temp( TI-1914)', 'Off Spec LPG from CRU inlet pressure']
MODEL : Random Forest (Low Importance Features Removed)
Features Used    : 13
Total Samples    : 458
Training Samples : 366
Testing Samples  : 92

Training Random Forest...

Performance
-----------------------------------
MAE  : 2.7229
RMSE : 3.5367
R²   : 0.7469


BASELINE COMPARISON
Previous Best
MAE  : 2.7091
RMSE : 3.6398
R²   : 0.7320

Current Model
MAE  : 2.7229
RMSE : 3.5367
R²   : 0.7469

✅ Hypothesis Accepted
Removing low-importance features improved prediction.


# Experiment 10 — Final Model Validation

## Hypothesis

After removing the lowest permutation importance features, the Random Forest
model should generalize better across multiple train-test splits.

5-Fold Cross Validation is performed on the final reduced-feature model.

---

## Final Feature Set

Removed Features

- Stabilizer Bottom T
- Stab. 3rd Tray
- Bottom Reboiler Naphtha Inlet Temp (TI-1907)
- Bot. Naphtha Reboiler Outlet Temp (TI-1908 & TI-1909)

Remaining Features

13 Process Variables

---

## Validation

5-Fold Cross Validation

Metrics

- R²
- MAE
- RMSE

---

## Previous Cross Validation

Mean R² = 0.6711

---

## Success Criterion

If the reduced-feature model achieves a higher average cross-validation score,
the feature reduction strategy is accepted and the model is frozen.

In [14]:
import pandas as pd

df_clean = pd.read_csv("I:\Projects\iocl\data\df_clean.csv")

<>:3: SyntaxWarning: invalid escape sequence '\P'
<>:3: SyntaxWarning: invalid escape sequence '\P'
C:\Users\Vansh Pandey\AppData\Local\Temp\ipykernel_7212\1369208814.py:3: SyntaxWarning: invalid escape sequence '\P'
  df_clean = pd.read_csv("I:\Projects\iocl\data\df_clean.csv")


In [17]:
# ==========================================================
# Clean Dataset + 5-Fold Cross Validation
# ==========================================================

import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold, cross_validate

# ----------------------------------------------------------
# Create Clean Dataset
# ----------------------------------------------------------
df_clean = df.copy()

df_clean = df_clean.drop(columns=[
    "DATE",
    "Stabilizer Top T.1",
    "Stab. 3rd Tray.1",
    "Bot. Naphtha Reboiler Outlet Temp(TI-1908 & TI-1909).1",
    "Off Spec LPG flow (tag Faulty)"
], errors="ignore")

# ----------------------------------------------------------
# Features & Target
# ----------------------------------------------------------
X = df_clean.drop(columns=["LPG Flow", "Wthr"])
y = df_clean["LPG Flow"]

# ----------------------------------------------------------
# Verify
# ----------------------------------------------------------
print("="*60)
print("Dataset Check")
print("="*60)
print("X Shape :", X.shape)
print("y Shape :", y.shape)
print("\nObject Columns:")
print(X.select_dtypes(include="object").columns.tolist())

# ----------------------------------------------------------
# Random Forest Model
# ----------------------------------------------------------
model = RandomForestRegressor(random_state=42)

# ----------------------------------------------------------
# 5-Fold Cross Validation
# ----------------------------------------------------------
cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scores = cross_validate(
    estimator=model,
    X=X,
    y=y,
    cv=cv,
    scoring={
        "r2": "r2",
        "mae": "neg_mean_absolute_error",
        "rmse": "neg_root_mean_squared_error"
    },
    return_train_score=False,
    n_jobs=-1
)

# ----------------------------------------------------------
# Results
# ----------------------------------------------------------
print("\n" + "="*60)
print("5-Fold Cross Validation Results")
print("="*60)

print(f"Mean R²   : {scores['test_r2'].mean():.4f} ± {scores['test_r2'].std():.4f}")
print(f"Mean MAE  : {-scores['test_mae'].mean():.4f}")
print(f"Mean RMSE : {-scores['test_rmse'].mean():.4f}")

print("\nFold-wise R² Scores:")
print(np.round(scores["test_r2"], 4))

Dataset Check
X Shape : (458, 17)
y Shape : (458,)

Object Columns:
[]

5-Fold Cross Validation Results
Mean R²   : 0.6711 ± 0.0415
Mean MAE  : 2.9863
Mean RMSE : 4.1253

Fold-wise R² Scores:
[0.7188 0.6249 0.6889 0.6184 0.7042]
